# Lenguajes y Autómatas II (SCD-1016)
## Sesión 37 · Integración del Pipeline Completo del Compilador

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se pasa de una expresión en el código fuente a un archivo .s que el ensamblador acepta?
>
> **Objetivo de la sesión:** Emitir un archivo ensamblador mínimo pero completo: secciones, símbolo global, prólogo, cuerpo y epílogo, respetando el contrato que espera as.

**Entregable:** tu implementación de emitir_funcion() en artesano_sesion37_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion37.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes el nombre de una función y el valor entero que debe devolver; escribes en el buffer un ensamblador x86-64 completo con sección, símbolo global, prólogo, el valor en %rax y epílogo.

Al final de la sesión tendrás: **Una función que emite un archivo .s mínimo válido y comprobable, con símbolo global, prólogo, cuerpo y epílogo.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 37](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion37.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion37.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion37.h"

int emitir_funcion(const char *nombre, int retorno, char *salida, int cap);
```

- La salida empieza en .text y debe contener .globl <nombre>, <nombre>:, pushq %rbp, movq %rsp, %rbp, movq $<retorno>, %rax, leave y ret.
- Cada pieza va en su propia línea terminada en salto de línea.
- Si falta espacio en el buffer o algún puntero es NULL, no se escribe nada y se devuelve -1.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion37_c.c` junto con `artesano_sesion37_c.c`
> (o `ia_sesion37_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `emitir_funcion("main", 7)` | 0 |
| 2 | `salida .globl main` | 1 |
| 3 | `salida pushq %rbp` | 1 |
| 4 | `salida movq $7, %rax` | 1 |
| 5 | `emitir_funcion(NULL)` | -1 |
| 6 | `buffer de 16 bytes` | -1 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 37 - Conexion de Frontend, TAC y Backend ===
Caso                               esperado:              obtenido:              resultado
[ 1] emitir_funcion("main", 7)          esperado: 0            obtenido: 0            OK
[ 2] salida .globl main                 esperado: 1            obtenido: 1            OK
[ 3] salida pushq %rbp                  esperado: 1            obtenido: 1            OK
[ 4] salida movq $7, %rax               esperado: 1            obtenido: 1            OK
[ 5] emitir_funcion(NULL)               esperado: -1           obtenido: -1           OK
[ 6] buffer de 16 bytes                 esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion37.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] El buffer contiene .text, .globl <nombre> y la etiqueta <nombre>: antes del prólogo.
- [ ] El cuerpo carga el retorno en %rax y el epílogo usa leave y ret.
- [ ] Los seis casos del harness salen OK y el programa termina con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion37_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion37.h
/* api_sesion37.h - contrato de la Sesion 37
 * Conexión de Frontend, TAC y Backend
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION37_H
#define API_SESION37_H

int emitir_funcion(const char *nombre, int retorno, char *salida, int cap);

#endif /* API_SESION37_H */


In [ ]:
%%writefile artesano_sesion37_c.c
/* artesano_sesion37_c.c - Fase 1: El Artesano (sin IA)
 * Conexión de Frontend, TAC y Backend.
 *
 * Implementa lo que declara api_sesion37.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion37.h"
#include <stdio.h>
#include <string.h>

int emitir_funcion(const char *nombre, int retorno, char *salida, int cap) {
    (void)nombre;
    (void)retorno;
    (void)salida;
    (void)cap; /* TODO: emitir .text, .globl, prologo, cuerpo y epilogo */
    return 0;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion37_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion37_c.c
/* ia_sesion37_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion37.h. No puede traer main().
 */
#include "api_sesion37.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion37_c.c
/* harness_sesion37_c.c - auditoria automatica de la Sesion 37
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion37_c.c artesano_sesion37_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion37.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 37 - Conexion de Frontend, TAC y Backend ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { char b[512]; int r = emitir_funcion("main", 7, b, sizeof b); verificar_int("emitir_funcion(\"main\", 7)", (long)(0), (long)(r)); }
    { char b[512] = ""; emitir_funcion("main", 7, b, sizeof b); int r = strstr(b, ".globl main") != NULL; verificar_int("salida .globl main", (long)(1), (long)(r)); }
    { char b[512] = ""; emitir_funcion("main", 7, b, sizeof b); int r = strstr(b, "pushq %rbp") != NULL; verificar_int("salida pushq %rbp", (long)(1), (long)(r)); }
    { char b[512] = ""; emitir_funcion("main", 7, b, sizeof b); int r = strstr(b, "movq $7, %rax") != NULL; verificar_int("salida movq $7, %rax", (long)(1), (long)(r)); }
    { char b[512]; int r = emitir_funcion(NULL, 7, b, sizeof b); verificar_int("emitir_funcion(NULL)", (long)(-1), (long)(r)); }
    { char b[16]; int r = emitir_funcion("main", 7, b, sizeof b); verificar_int("buffer de 16 bytes", (long)(-1), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion37.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion37_c.c artesano_sesion37_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion37_c.c ia_sesion37_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion37.md` a `auditorias/auditoria_sesion37.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
